In [2]:
today_date="11-11-2025"

StatementMeta(, 00632eac-1ae2-4d5c-9a4b-dba432047f7b, 4, Finished, Available, Finished)

In [3]:
Fabric_bronze_path='abfss://b0cceb94-d226-4dbd-a75b-789c5defa0a2@onelake.dfs.fabric.microsoft.com/f5ac5edb-ff4b-4a7b-ab7c-1b4182e0d30b/Tables/tblsales_bronze'
from pyspark.sql.functions import col
df=spark.read.format('delta').load(Fabric_bronze_path)

StatementMeta(, 00632eac-1ae2-4d5c-9a4b-dba432047f7b, 5, Finished, Available, Finished)

In [4]:
display(df)


StatementMeta(, 00632eac-1ae2-4d5c-9a4b-dba432047f7b, 6, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 3729602e-8f65-4e48-93a3-837523f99dcd)

### Creating columns for Statistics and dropping the useless ones

In [5]:
df_new=df.drop("UOM_ID","SCALAR_FACTOR","SCALAR_ID","VECTOR","COORDINATE","STATUS","SYMBOL","TERMINATED","DECIMALS")

StatementMeta(, 00632eac-1ae2-4d5c-9a4b-dba432047f7b, 7, Finished, Available, Finished)

In [12]:
df_new.select("REF_DATE").distinct().show()
df_new.select("Statistics").distinct().show()
df_new.select("Violations").distinct().count()

StatementMeta(, 00632eac-1ae2-4d5c-9a4b-dba432047f7b, 14, Finished, Available, Finished)

+----------+
|  REF_DATE|
+----------+
|2023-01-01|
|2020-01-01|
|2021-01-01|
|2022-01-01|
|2024-01-01|
+----------+

+--------------------+
|          Statistics|
+--------------------+
|Total, adult charged|
|Rate, youth charg...|
|Rate per 100,000 ...|
|   Cleared otherwise|
|Total, persons ch...|
|       Total cleared|
|Rate, total perso...|
|    Actual incidents|
|Total, youth charged|
|Percentage change...|
|   Percent unfounded|
| Unfounded incidents|
|Rate, youth not c...|
|Rate, adult charg...|
|Total, youth not ...|
|   Cleared by charge|
+--------------------+



312

In [ ]:
from pyspark.sql import functions as F
df_pivot = (
    df_new
    .groupBy("REF_DATE", "GEO", "DGUID", "Violations")  # group by identifying columns
    .pivot("Statistics")  # make each unique "Statistics" value a column
    .agg(F.first("VALUE"))  # take the first (or you can use sum, avg, etc.)
)

display(df_pivot)

StatementMeta(, f6b8e738-68fb-44b2-acda-88dbefe1b160, 8, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 469b161d-fa2b-4477-90fc-a0c929ba75b4)

### Handling Missing Values

In [ ]:
df_filled = df_pivot.fillna(0.0)


StatementMeta(, f6b8e738-68fb-44b2-acda-88dbefe1b160, 11, Finished, Available, Finished)

In [ ]:
from pyspark.sql import functions as F
from functools import reduce


# Identify statistic columns
stat_cols = [c for c in df_filled.columns if c not in ["REF_DATE", "GEO", "DGUID", "Violations"]]

#Aggregate stats across years per Violation
df_agg = (
    df_filled
    .groupBy("Violations")
    .agg(*[F.sum(F.col(c)).alias(c) for c in stat_cols])  # sum across all years
)

#Filter Violations where all stats are 0
df_zero_all_years = df_agg.filter(reduce(lambda a, b: a & b, [F.col(c) == 0 for c in stat_cols]))

display(df_zero_all_years)

StatementMeta(, f6b8e738-68fb-44b2-acda-88dbefe1b160, 26, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 19b09b75-5f86-41af-8c42-4fe800601d3d)

In [ ]:
zero_violations = [row["Violations"] for row in df_zero_all_years.collect()]

# 5️⃣ Filter the original df_pivot to exclude those violations
df_cleaned = df_pivot.filter(~F.col("Violations").isin(zero_violations))

StatementMeta(, f6b8e738-68fb-44b2-acda-88dbefe1b160, 27, Finished, Available, Finished)

In [ ]:
print("Original rows:", df_pivot.count())
print("Cleaned rows:", df_cleaned.count())
print("Longer dataset:", df.count())

StatementMeta(, f6b8e738-68fb-44b2-acda-88dbefe1b160, 28, Finished, Available, Finished)

Original rows: 686400
Cleaned rows: 616000
Longer dataset: 10982400


In [ ]:
df_cleaned.drop_duplicates()

StatementMeta(, f6b8e738-68fb-44b2-acda-88dbefe1b160, 29, Finished, Available, Finished)

DataFrame[REF_DATE: date, GEO: string, DGUID: string, Violations: string, Actual incidents: double, Cleared by charge: double, Cleared otherwise: double, Percent unfounded: double, Percentage change in rate: double, Rate per 100,000 population: double, Rate, adult charged per 100,000 population aged 18 years and over: double, Rate, total persons charged per 100,000 population aged 12 years and over: double, Rate, youth charged per 100,000 population aged 12 to 17 years: double, Rate, youth not charged per 100,000 population aged 12 to 17 years: double, Total cleared: double, Total, adult charged: double, Total, persons charged: double, Total, youth charged: double, Total, youth not charged: double, Unfounded incidents: double]

In [ ]:
print("Cleaned rows:", df_cleaned.count())

StatementMeta(, f6b8e738-68fb-44b2-acda-88dbefe1b160, 30, Finished, Available, Finished)

Cleaned rows: 616000


### Writing to Silver Table

In [ ]:
df_cleaned.createOrReplaceTempView('t_silver_new_data')

StatementMeta(, f6b8e738-68fb-44b2-acda-88dbefe1b160, 31, Finished, Available, Finished)

In [ ]:
%%sql
SELECT * from t_silver_new_data

StatementMeta(, f6b8e738-68fb-44b2-acda-88dbefe1b160, 32, Finished, Available, Finished)

<Spark SQL result set with 1000 rows and 20 fields>

In [ ]:
Fabric_tblsales_silver="abfss://b0cceb94-d226-4dbd-a75b-789c5defa0a2@onelake.dfs.fabric.microsoft.com/f5ac5edb-ff4b-4a7b-ab7c-1b4182e0d30b/Tables/tblsales_silver"
try:
    spark.read.format('delta').load(Fabric_tblsales_silver).createOrReplaceTempView('t_tblsales_silver')
except:
    v_create_table = f"""CREATE TABLE IF NOT EXISTS tblsales_silver (
    REF_DATE DATE,
    GEO STRING,
    DGUID STRING,
    Violations STRING,
    Actual_incidents INT,
    Cleared_by_charge INT,
    Cleared_otherwise INT,
    Percent_unfounded DOUBLE,
    Percentage_change_in_rate DOUBLE,
    Rate_per_100000_population DOUBLE,
    Rate_adult_charged_per_100000_population_18_plus DOUBLE,
    Rate_total_persons_charged_per_100000_population_12_plus DOUBLE,
    Rate_youth_charged_per_100000_population_12_to_17 DOUBLE,
    Rate_youth_not_charged_per_100000_population_12_to_17 DOUBLE,
    Total_cleared INT,
    Total_adult_charged INT,
    Total_persons_charged INT,
    Total_youth_charged INT,
    Total_youth_not_charged INT,
    Unfounded_incidents INT)
    USING DELTA
    """
    spark.sql(v_create_table)
    spark.read.format('delta').load(Fabric_tblsales_silver).createOrReplaceTempView('t_tblsales_silver')

StatementMeta(, f6b8e738-68fb-44b2-acda-88dbefe1b160, 33, Finished, Available, Finished)

In [ ]:
sql_statement=f'''MERGE INTO tblsales_silver AS target
USING t_silver_new_data AS source
ON target.DGUID = source.DGUID
   AND target.REF_DATE = source.REF_DATE
   AND target.Violations = source.Violations
WHEN MATCHED THEN
    UPDATE SET
        target.GEO = source.GEO,
        target.Actual_incidents = source.`Actual incidents`,
        target.Cleared_by_charge = source.`Cleared by charge`,
        target.Cleared_otherwise = source.`Cleared otherwise`,
        target.Percent_unfounded = source.`Percent unfounded`,
        target.Percentage_change_in_rate = source.`Percentage change in rate`,
        target.Rate_per_100000_population = source.`Rate per 100,000 population`,
        target.Rate_adult_charged_per_100000_population_18_plus = source.`Rate, adult charged per 100,000 population aged 18 years and over`,
        target.Rate_total_persons_charged_per_100000_population_12_plus = source.`Rate, total persons charged per 100,000 population aged 12 years and over`,
        target.Rate_youth_charged_per_100000_population_12_to_17 = source.`Rate, youth charged per 100,000 population aged 12 to 17 years`,
        target.Rate_youth_not_charged_per_100000_population_12_to_17 = source.`Rate, youth not charged per 100,000 population aged 12 to 17 years`,
        target.Total_cleared = source.`Total cleared`,
        target.Total_adult_charged = source.`Total, adult charged`,
        target.Total_persons_charged = source.`Total, persons charged`,
        target.Total_youth_charged = source.`Total, youth charged`,
        target.Total_youth_not_charged = source.`Total, youth not charged`,
        target.Unfounded_incidents = source.`Unfounded incidents`
WHEN NOT MATCHED THEN
    INSERT (
        REF_DATE, GEO, DGUID, Violations, Actual_incidents, Cleared_by_charge, Cleared_otherwise,
        Percent_unfounded, Percentage_change_in_rate, Rate_per_100000_population,
        Rate_adult_charged_per_100000_population_18_plus, Rate_total_persons_charged_per_100000_population_12_plus,
        Rate_youth_charged_per_100000_population_12_to_17, Rate_youth_not_charged_per_100000_population_12_to_17,
        Total_cleared, Total_adult_charged, Total_persons_charged, Total_youth_charged,
        Total_youth_not_charged, Unfounded_incidents
    )
    VALUES (
        source.REF_DATE, source.GEO, source.DGUID, source.`Violations`, source.`Actual incidents`,
        source.`Cleared by charge`, source.`Cleared otherwise`, source.`Percent unfounded`,
        source.`Percentage change in rate`, source.`Rate per 100,000 population`,
        source.`Rate, adult charged per 100,000 population aged 18 years and over`,
        source.`Rate, total persons charged per 100,000 population aged 12 years and over`,
        source.`Rate, youth charged per 100,000 population aged 12 to 17 years`,
        source.`Rate, youth not charged per 100,000 population aged 12 to 17 years`,
        source.`Total cleared`, source.`Total, adult charged`, source.`Total, persons charged`,
        source.`Total, youth charged`, source.`Total, youth not charged`, source.`Unfounded incidents`  
    );
'''
spark.sql(sql_statement).show()

StatementMeta(, f6b8e738-68fb-44b2-acda-88dbefe1b160, 35, Finished, Available, Finished)

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|           616000|          616000|               0|                0|
+-----------------+----------------+----------------+-----------------+

